# PA3 — treinos pesados no Colab (GPU)

Roda no Colab o que é caro em CPU: o **MegaDetector**, o **professor com 3 recortes**
(BioCLIP 2 nas laterais da foto), os **alunos da Parte 1**, a **Parte 2** (curva de
orçamento) e, no fim, o **modelo final** com a galeria da Parte 3.

**Antes de começar**
1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.
2. O `pa3_dados.zip` (415 MB) precisa estar no Google Drive em `MeuDrive/PA3/pa3_dados.zip`.
3. `Ambiente de execução → Executar tudo`.

**Rodadas já feitas não são refeitas.** Os resultados ficam em `MeuDrive/PA3/run/`, e cada
script pula o que já existe lá. Num Drive vazio, "Executar tudo" refaz o pipeline do Colab
(~2h30 numa T4):
- célula 7: MegaDetector no train e no test_public (~5 min);
- célula 7b: recortes laterais do train + pseudo-rótulos do professor de 3 recortes (~15 min);
- célula 8: os 10 alunos da Parte 1 e a escolha pelo val (~1 h);
- célula 9: piso e aluno escolhido nos orçamentos 1/5/20 (~1 h);
- célula 10: modelo final, conferência com o `predict.py`, figuras e o zip (~5 min).

**Se o Colab desconectar:** rode tudo de novo. Checkpoints (a cada época) ficam no Drive, e
cada script continua de onde parou.

**No fim:** a última célula baixa `outputs_colab.zip`. Coloque na pasta do PA3 local e avise.

In [ ]:
# 1. GPU disponível?
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução -> Alterar o tipo -> GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive (dados de entrada e persistência dos resultados)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PA3'
RUN = f'{DRIVE}/run'
import os
assert os.path.exists(f'{DRIVE}/pa3_dados.zip'), f'Suba o pa3_dados.zip para {DRIVE}/'
os.makedirs(f'{RUN}/outputs', exist_ok=True); os.makedirs(f'{RUN}/checkpoints', exist_ok=True)

In [ ]:
# 3. Código: clona (ou atualiza) o repositório
REPO = '/content/PA3_Deep_learning'
if not os.path.exists(REPO):
    !git clone -q https://github.com/matheuspinto4/PA3_Deep_learning.git {REPO}
else:
    !cd {REPO} && git fetch -q && git checkout -q -- . && git pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Dados: descompacta no disco local do Colab (bem mais rápido que ler do Drive)
if not os.path.exists(f'{REPO}/pa3_dados/pa3_dados/LEIAME.md'):
    !mkdir -p {REPO}/pa3_dados && unzip -q -o {DRIVE}/pa3_dados.zip -d {REPO}/pa3_dados
    # o zip pode ou não ter a pasta pa3_dados/ no topo: normaliza para pa3_dados/pa3_dados/
    if not os.path.exists(f'{REPO}/pa3_dados/pa3_dados/LEIAME.md'):
        !mkdir -p /tmp/pa3x && mv {REPO}/pa3_dados/* /tmp/pa3x/ && mkdir -p {REPO}/pa3_dados/pa3_dados && mv /tmp/pa3x/* {REPO}/pa3_dados/pa3_dados/
!ls {REPO}/pa3_dados/pa3_dados/images && ls {REPO}/pa3_dados/pa3_dados/images/train | wc -l

In [ ]:
# 5. Persistência: outputs/ e checkpoints/ do repo passam a apontar para o Drive.
#    Na 1a vez, copia para o Drive os resultados já versionados (Partes 0 e 1);
#    -n = não sobrescreve o que já existe lá (resultados de rodadas anteriores).
import pathlib
if not os.path.islink(f'{REPO}/outputs'):
    !cp -rn {REPO}/outputs/. {RUN}/outputs/
    !rm -rf {REPO}/outputs && ln -s {RUN}/outputs {REPO}/outputs
if not os.path.islink(f'{REPO}/checkpoints'):
    !rm -rf {REPO}/checkpoints && ln -s {RUN}/checkpoints {REPO}/checkpoints
!ls -la {REPO} | grep -E 'outputs|checkpoints'

In [ ]:
# 6. Dependências extras (o resto já vem no Colab) + cache de imagens 224x320 (~2-4 min)
!pip install -q ultralytics==8.4.206 open_clip_torch==4.2.1
!python scripts/build_cache.py

In [ ]:
# 7. MegaDetector V6 no test_public e no train: tamanho dos animais (~5 min)
!python -u scripts/part1_megadetector.py

In [ ]:
# 7b. Professor com 3 recortes: embeddings das laterais do train (GPU), escolha da
#     combinação nos 105 rótulos, pseudo-rótulos novos (orçamentos 1/5/20) (~10-15 min)
!set -o pipefail; python -u scripts/part1_teacher_multicrop.py 2>&1 | tee outputs/part1/teacher_multicrop_log_colab.txt; echo $? > /tmp/rc
assert open('/tmp/rc').read().strip() == '0', 'a célula falhou: veja o erro acima (Executar tudo para aqui)'

In [ ]:
# 8. Parte 1: os alunos (orçamento 5, seed 0) -- 6 com o professor de center crop (já
#    treinados, só relidos) e 4 com o professor de 3 recortes -- e a escolha pelo VAL (~25 min)
!set -o pipefail; python -u scripts/part1_student.py 2>&1 | tee outputs/part1/student_log_colab.txt; echo $? > /tmp/rc
assert open('/tmp/rc').read().strip() == '0', 'a célula falhou: veja o erro acima (Executar tudo para aqui)'

In [ ]:
# 9. Parte 2: piso e aluno escolhido nos orçamentos 1/5/20 (3 seeds no 5 e no 20).
#    Só treina o que falta no Drive (se o aluno escolhido mudou, são 6 rodadas novas dele) (~35 min)
!set -o pipefail; python -u scripts/part2_budget.py 2>&1 | tee outputs/part2_log_colab.txt; echo $? > /tmp/rc
assert open('/tmp/rc').read().strip() == '0', 'a célula falhou: veja o erro acima (Executar tudo para aqui)'

In [ ]:
# 10. Modelo final: exporta o aluno escolhido, confere com o predict.py (F1 tem que bater com
#     o do treino), refaz as análises que dependem dele e empacota os resultados para baixar
import json
chosen = json.load(open('outputs/part1/student_results.json'))['chosen']
print('modelo final:', chosen)
FINAL = 'outputs/final_model/modelo_final.pt'
!python scripts/export_model.py checkpoints/student/{chosen}_b5_s0.pt --saida {FINAL} --origem "aluno {chosen}, orçamento 5, seed 0 (Colab)"
!python predict.py --imagens pa3_dados/pa3_dados/images/test_public --saida outputs/final_model/preds_test_public.csv --pesos {FINAL} --rotulos pa3_dados/pa3_dados/labels/test_public.csv | tr '\r' '\n' | grep -v '^  [0-9]'
print('F1 do treino:', json.load(open(f'outputs/part1/students/{chosen}_b5_s0.json'))['test_f1_macro'])
!python -u scripts/part1_size_analysis.py
!python -u scripts/part1_cottontail.py
!python -u scripts/part3_gallery.py --pesos {FINAL} 2>&1 | tail -15
!cd {RUN} && rm -f outputs_colab.zip && zip -qr outputs_colab.zip outputs && ls -la outputs_colab.zip
from google.colab import files
files.download(f'{RUN}/outputs_colab.zip')